# Reel Insights: TMDB Movie Data Analysis

## 1. Project Overview

This project focuses on Exploratory Data Analysis (EDA) of the TMDB movie dataset.

The objective is to explore movie industry trends, analyze financial performance,
identify popular genres, and understand factors influencing movie ratings.

## 2. Data Cleaning

In this section, we will inspect and clean the six CSV datasets before performing
SQL analysis, Pandas analysis, and data visualization.

## 3. Importing Libraries and Loading Dataset

In [17]:
import pandas as pd
import numpy as np

In [3]:
movies_df = pd.read_csv(r"data/movies.csv")
movies_df.head()

,movie_id,title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,status,overview
0,157336,Interstellar,en,2014-11-05,169,165000000.0,7.466067e+08,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...
1,27205,Inception,en,2010-07-15,148,160000000.0,8.390306e+08,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es..."
2,24428,The Avengers,en,2012-04-25,143,220000000.0,1.518816e+09,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...
3,155,The Dark Knight,en,2008-07-16,152,185000000.0,1.004558e+09,60.5510,8.534,36255,Released,Batman raises the stakes in his war on crime. ...
4,19995,Avatar,en,2009-12-16,162,237000000.0,2.923706e+09,48.1996,7.610,34348,Released,"In the 22nd century, a paraplegic Marine is di..."


In [5]:
movies_df.shape

(2509, 12)

In [6]:
movies_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2509 entries, 0 to 2508
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movie_id           2509 non-null   int64  
 1   title              2509 non-null   str    
 2   original_language  2509 non-null   str    
 3   release_date       2509 non-null   str    
 4   runtime            2509 non-null   int64  
 5   budget             2509 non-null   float64
 6   revenue            2509 non-null   float64
 7   popularity         2509 non-null   float64
 8   vote_average       2509 non-null   float64
 9   vote_count         2509 non-null   int64  
 10  status             2509 non-null   str    
 11  overview           2509 non-null   str    
dtypes: float64(4), int64(3), str(5)
memory usage: 989.6 KB


### Missing Values Inspection

- Total records: 2,509
- Total columns: 12
- No missing values were found in any column.
- All columns contain 2,509 non-null values.

**Observation:** The movies.csv dataset does not contain any missing values.

### Release Date Data Type Conversion

- The `release_date` column is currently stored as a string (`str`).
- We need to convert it into a datetime format.
- This will help us perform date-based analysis, such as yearly movie releases and release trends.

In [7]:
movies_df['release_date'] = pd.to_datetime(
  movies_df['release_date'],
  errors = 'coerce'
)

In [8]:
movies_df['release_date'].dtype

dtype('<M8[us]')

In [9]:
movies_df['release_date'].isna().sum()

np.int64(0)

**Observation:**
- The `release_date` column was successfully converted from string to datetime.
- No invalid dates were found after conversion.
- Total missing/invalid dates: 0.

### Duplicate Records Inspection

- Check whether the dataset contains completely duplicated rows.
- Identify the number of duplicate records.
- Decide whether duplicate records need to be removed.

In [10]:
movies_df.duplicated().sum()

np.int64(6)

**Observation:**
- A total of 6 completely duplicated rows were identified.
- These duplicate records need to be removed to avoid counting the same movie records multiple times.

In [11]:
movies_df = movies_df.drop_duplicates()

In [12]:
movies_df.duplicated().sum()

np.int64(0)

**Observation:**
- Initially, 6 completely duplicated records were identified.
- Duplicate records were removed using Pandas `drop_duplicates()`.
- The dataset now contains no completely duplicated rows.
- The removal of duplicates helps prevent repeated records from affecting further analysis.

In [13]:
movies_df.shape

(2503, 12)

### Zero Values Inspection – Budget and Revenue

In [14]:
# Count zero values in budget
(movies_df['budget'] == 0).sum()

np.int64(90)

In [15]:
# Count zero values in revenue
(movies_df['revenue'] == 0).sum()

np.int64(102)

### Zero Values Inspection – Budget and Revenue

**Observation:**
- 90 movies have a budget value of 0.
- 102 movies have a revenue value of 0.
- These zero values may represent unavailable financial information rather than actual zero values.

**Data Cleaning Decision:**
- We will retain all movie records.
- We will treat zero budget and revenue values as unavailable (`NaN`) for financial analysis.
- These records will be excluded from relevant financial calculations where appropriate.

In [16]:
((movies_df['budget'] == 0) & (movies_df['revenue'] == 0)).sum()

np.int64(59)

In [18]:
movies_df['budget'] = movies_df['budget'].replace(0, np.nan)

In [19]:
movies_df['revenue'] = movies_df['revenue'].replace(0, np.nan)

### Handling Zero Values in Budget and Revenue

**Observation:**
- 90 movies have zero budget.
- 102 movies have zero revenue.
- 59 movies have both budget and revenue equal to zero.

**Data Cleaning Approach:**
- Replace zero values in `budget` and `revenue` with `NaN`.
- Calculate the median of each financial column separately.
- Replace missing values with their respective column median.
- Retain all movie records.

In [21]:
movies_df['budget'].median()

np.float64(40000000.0)

In [22]:
movies_df['revenue'].median()

np.float64(136300000.0)

In [23]:
movies_df['budget'] = movies_df['budget'].fillna(40000000.0)

In [24]:
movies_df['revenue'] = movies_df['revenue'].fillna(136300000.0)

In [25]:
movies_df[['budget', 'revenue']].isna().sum()

budget     0
revenue    0
dtype: int64

**Verification:**
- All missing values in the `budget` column were filled using its median (40,000,000).
- All missing values in the `revenue` column were filled using its median (136,300,000).
- Verified that both columns contain zero missing values after imputation.
- All movie records have been retained.

In [26]:
movies_df.isna().sum()

movie_id             0
title                0
original_language    0
release_date         0
runtime              0
budget               0
revenue              0
popularity           0
vote_average         0
vote_count           0
status               0
overview             0
dtype: int64

In [27]:
movies_df.info()

<class 'pandas.DataFrame'>
Index: 2503 entries, 0 to 2508
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   movie_id           2503 non-null   int64         
 1   title              2503 non-null   str           
 2   original_language  2503 non-null   str           
 3   release_date       2503 non-null   datetime64[us]
 4   runtime            2503 non-null   int64         
 5   budget             2503 non-null   float64       
 6   revenue            2503 non-null   float64       
 7   popularity         2503 non-null   float64       
 8   vote_average       2503 non-null   float64       
 9   vote_count         2503 non-null   int64         
 10  status             2503 non-null   str           
 11  overview           2503 non-null   str           
dtypes: datetime64[us](1), float64(4), int64(3), str(4)
memory usage: 983.2 KB


### Data Type Verification & Final Cleaning Summary

**Data Type Verification:**
- `release_date` was successfully converted from string to datetime.
- Numeric columns have appropriate integer or floating-point data types.
- Text columns are stored as strings.
- No further data type corrections were required.

**Final Cleaning Summary:**
- Original records: 2,509.
- Duplicate records removed: 6.
- Final records: 2,503.
- Zero budget and revenue values were replaced with `NaN` and filled using their respective column medians.
- All 12 columns were checked, and no missing values remain.

**Conclusion:** The `movies_df` dataset is successfully cleaned and ready for further analysis.

In [28]:
genres_df = pd.read_csv(r"data/genres.csv")
genres_df.head()

,genre_id,genre_name
0,28,Action
1,12,Adventure
2,16,Animation
3,35,Comedy
4,80,Crime


In [29]:
genres_df.shape

(19, 2)

In [30]:
genres_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   genre_id    19 non-null     int64
 1   genre_name  19 non-null     str  
dtypes: int64(1), str(1)
memory usage: 573.0 bytes


In [31]:
genres_df.isna().sum()

genre_id      0
genre_name    0
dtype: int64

In [32]:
genres_df.duplicated().sum()

np.int64(0)

### Genres Dataset – Cleaning Summary

**Observations:**
- The dataset contains 19 records and 2 columns.
- `genre_id` is stored as an integer and `genre_name` as a string.
- No missing values were found.
- No duplicate records were identified.
- Both columns have appropriate data types.

**Conclusion:** The genres dataset is clean and ready for further analysis.

In [33]:
movie_genres_df = pd.read_csv(r"data/movie_genres.csv")
movie_genres_df.head()

,movie_id,genre_id
0,157336,12
1,157336,18
2,157336,878
3,27205,28
4,27205,878


In [34]:
movie_genres_df.shape

(6888, 2)

In [35]:
movie_genres_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6888 entries, 0 to 6887
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   movie_id  6888 non-null   int64
 1   genre_id  6888 non-null   int64
dtypes: int64(2)
memory usage: 107.8 KB


In [36]:
movie_genres_df.isna().sum()

movie_id    0
genre_id    0
dtype: int64

In [37]:
movie_genres_df.duplicated().sum()

np.int64(10)

In [40]:
movie_genres_df = movie_genres_df.drop_duplicates()

In [41]:
movie_genres_df.duplicated().sum()

np.int64(0)

In [42]:
movie_genres_df.shape

(6878, 2)

### Movie Genres Dataset – Cleaning Summary

**Observations:**
- The dataset originally contained 6,888 records and 2 columns.
- Both `movie_id` and `genre_id` have appropriate integer data types.
- No missing values were found.
- 10 duplicate movie-genre combinations were identified and removed.
- The cleaned dataset contains 6,878 records and 2 columns.
- Verified that no duplicate records remain.

**Conclusion:** The `movie_genres_df` dataset is successfully cleaned and ready for further analysis.

In [43]:
(~movie_genres_df['movie_id'].isin(movies_df['movie_id'])).sum()

np.int64(0)

In [44]:
(~movie_genres_df['genre_id'].isin(genres_df['genre_id'])).sum()

np.int64(0)

### Referential Integrity Check

**Observations:**
- Checked whether all `movie_id` values in `movie_genres_df` exist in `movies_df`.
- Checked whether all `genre_id` values in `movie_genres_df` exist in `genres_df`.
- No unmatched movie IDs were found.
- No unmatched genre IDs were found.

**Conclusion:** All movie and genre references are valid. The `movie_genres_df` dataset maintains referential integrity with the `movies_df` and `genres_df` datasets.

In [45]:
cast_df = pd.read_csv(r"data/cast.csv")
cast_df.head()

,cast_row_id,movie_id,person_id,actor_name,character_name,cast_order
0,1,157336,10297,Matthew McConaughey,Cooper,0
1,2,157336,1813,Anne Hathaway,Brand,1
2,3,157336,3895,Michael Caine,Professor Brand,2
3,4,157336,83002,Jessica Chastain,Murph,3
4,5,157336,1893,Casey Affleck,Tom,4


In [46]:
cast_df.shape

(24917, 6)

In [47]:
cast_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24917 entries, 0 to 24916
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   cast_row_id     24917 non-null  int64
 1   movie_id        24917 non-null  int64
 2   person_id       24917 non-null  int64
 3   actor_name      24917 non-null  str  
 4   character_name  24906 non-null  str  
 5   cast_order      24917 non-null  int64
dtypes: int64(4), str(2)
memory usage: 1.7 MB


In [48]:
cast_df.isna().sum()

cast_row_id        0
movie_id           0
person_id          0
actor_name         0
character_name    11
cast_order         0
dtype: int64

In [49]:
cast_df[cast_df['character_name'].isna()]

,cast_row_id,movie_id,person_id,actor_name,character_name,cast_order
14465,14466,97370,1245,Scarlett Johansson,NaN,0
14466,14467,97370,1047644,Jeremy McWilliams,NaN,1
14467,14468,97370,1711557,Lynsey Taylor Mackay,NaN,2
14468,14469,97370,1711559,Dougie McConnell,NaN,3
14469,14470,97370,1711561,Kevin McAlinden,NaN,4
14470,14471,97370,1711616,D. Meade,NaN,5
14471,14472,97370,74841,Andrew Gorman,NaN,6
14472,14473,97370,1711626,Joe Szula,NaN,7
14473,14474,97370,136796,Kryštof Hádek,NaN,8
14474,14475,97370,1711628,Roy Armstrong,NaN,9


In [50]:
cast_df['character_name'] = cast_df['character_name'].fillna('Unknown')

In [51]:
cast_df.isna().sum()

cast_row_id       0
movie_id          0
person_id         0
actor_name        0
character_name    0
cast_order        0
dtype: int64

In [52]:
cast_df.duplicated().sum()

np.int64(15)

In [53]:
cast_df = cast_df.drop_duplicates()

In [54]:
print("Shape:", cast_df.shape)
print("Remaining duplicates:", cast_df.duplicated().sum())

Shape: (24902, 6)
Remaining duplicates: 0


In [55]:
(~cast_df['movie_id'].isin(movies_df['movie_id'])).sum()

np.int64(0)

### Cast Dataset – Cleaning Summary

**Observations:**
- The original dataset contained 24,917 records and 6 columns.
- 11 missing values were found in `character_name` and replaced with `"Unknown"`.
- All columns have appropriate data types.
- 15 duplicate records were identified and removed.
- The cleaned dataset contains 24,902 records and 6 columns.
- No missing values or duplicate records remain.
- All `movie_id` values were verified against `movies_df`; no unmatched references were found.

**Conclusion:** The `cast_df` dataset is successfully cleaned and maintains referential integrity with the movies dataset.

In [56]:
crew_df = pd.read_csv(r"data/crew.csv")
crew_df.head()

,crew_row_id,movie_id,person_id,person_name,job,department
0,1,157336,527,Jonathan Nolan,Writer,Writing
1,2,157336,525,Christopher Nolan,Writer,Writing
2,3,157336,525,Christopher Nolan,Director,Directing
3,4,27205,525,Christopher Nolan,Writer,Writing
4,5,27205,525,Christopher Nolan,Director,Directing


In [57]:
crew_df.shape

(7297, 6)

In [58]:
crew_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7297 entries, 0 to 7296
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   crew_row_id  7297 non-null   int64
 1   movie_id     7297 non-null   int64
 2   person_id    7297 non-null   int64
 3   person_name  7297 non-null   str  
 4   job          7297 non-null   str  
 5   department   7297 non-null   str  
dtypes: int64(3), str(3)
memory usage: 552.0 KB


In [59]:
crew_df.isna().sum()

crew_row_id    0
movie_id       0
person_id      0
person_name    0
job            0
department     0
dtype: int64

In [60]:
crew_df.duplicated().sum()

np.int64(8)

In [61]:
crew_df = crew_df.drop_duplicates()

In [62]:
print("Shape:", crew_df.shape)
print("Remaining duplicates:", crew_df.duplicated().sum())

Shape: (7289, 6)
Remaining duplicates: 0


In [63]:
(~crew_df['movie_id'].isin(movies_df['movie_id'])).sum()

np.int64(0)

### Crew Dataset – Cleaning Summary

**Observations:**
- The original dataset contained 7,297 records and 6 columns.
- No missing values were found.
- All columns have appropriate data types.
- 8 duplicate records were identified and removed.
- The cleaned dataset contains 7,289 records and 6 columns.
- No duplicate records remain.
- All `movie_id` values were verified against `movies_df`; no unmatched references were found.

**Conclusion:** The `crew_df` dataset is successfully cleaned and maintains referential integrity with the movies dataset.

In [64]:
movie_keywords_df = pd.read_csv(r"data/movie_keywords.csv")
movie_keywords_df.head()

,movie_id,keyword_id,keyword_name
0,157336,1612,spacecraft
1,157336,4776,race against time
2,157336,310,artificial intelligence (a.i.)
3,157336,1432,nasa
4,157336,1521,time warp


In [65]:
movie_keywords_df.shape

(41701, 3)

In [66]:
movie_keywords_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41701 entries, 0 to 41700
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   movie_id      41701 non-null  int64
 1   keyword_id    41701 non-null  int64
 2   keyword_name  41701 non-null  str  
dtypes: int64(2), str(1)
memory usage: 1.4 MB


In [67]:
movie_keywords_df.isna().sum()

movie_id        0
keyword_id      0
keyword_name    0
dtype: int64

In [68]:
movie_keywords_df.duplicated().sum()

np.int64(20)

In [69]:
movie_keywords_df = movie_keywords_df.drop_duplicates()

In [70]:
print("Shape:", movie_keywords_df.shape)
print("Remaining duplicates:", movie_keywords_df.duplicated().sum())

Shape: (41681, 3)
Remaining duplicates: 0


In [71]:
(~movie_keywords_df['movie_id'].isin(movies_df['movie_id'])).sum()

np.int64(0)

### Movie Keywords Dataset – Cleaning Summary

**Observations:**
- The original dataset contained 41,701 records and 3 columns.
- No missing values were found.
- All columns have appropriate data types.
- 20 duplicate records were identified and removed.
- The cleaned dataset contains 41,681 records and 3 columns.
- No duplicate records remain.
- All `movie_id` values were verified against `movies_df`; no unmatched references were found.

**Conclusion:** The `movie_keywords_df` dataset is successfully cleaned and maintains referential integrity with the movies dataset.

In [73]:
movies_df.to_csv("cleaned_data/movies.csv", index=False)
genres_df.to_csv("cleaned_data/genres.csv", index=False)
movie_genres_df.to_csv("cleaned_data/movie_genres.csv", index=False)
cast_df.to_csv("cleaned_data/cast.csv", index=False)
crew_df.to_csv("cleaned_data/crew.csv", index=False)
movie_keywords_df.to_csv("cleaned_data/movie_keywords.csv", index=False)

print("All six cleaned datasets saved successfully!")

All six cleaned datasets saved successfully!


In [75]:
cleaned_files = {
    "movies": "cleaned_data/movies.csv",
    "genres": "cleaned_data/genres.csv",
    "movie_genres": "cleaned_data/movie_genres.csv",
    "cast": "cleaned_data/cast.csv",
    "crew": "cleaned_data/crew.csv",
    "movie_keywords": "cleaned_data/movie_keywords.csv"
}

for name, path in cleaned_files.items():
    df = pd.read_csv(path)
    print(f"{name}: {df.shape}")

movies: (2503, 12)
genres: (19, 2)
movie_genres: (6878, 2)
cast: (24902, 6)
crew: (7289, 6)
movie_keywords: (41681, 3)
